# Churn

Logo churn and revenue churn from the Tidemill engine.

## How churn is measured

**Logo (customer) churn** — the fraction of *customers* who fully cancel:

```
logo_churn = customers_fully_churned / customers_active_at_period_start
```

A customer is "active at start" if they had positive MRR at the period
start (cumulative MRR movements before start > 0).  They are "fully
churned" if, by the period end, they have zero active subscriptions
remaining.

**Revenue churn** — the fraction of *MRR* lost to cancellations:

```
revenue_churn = |churned_MRR| / starting_MRR
```

`churned_MRR` sums the MRR of every subscription whose churn event
falls within `[start, end)`.

In [1]:
from tidemill import reports
from tidemill.reports.client import TidemillClient

reports.setup()

START, END = "2025-08-01", "2026-03-31"
CHURN_START = "2025-08-01"
tm = TidemillClient()

## 1. Customer churn sets

**C_start** — customers active at period start (positive MRR).
**C_churned** — subset of C_start who fully churned during the window.

In [2]:
detail = reports.churn.customer_detail(tm, CHURN_START, END)
reports.churn.style_c_start(detail)

customer,customer_name,starting_mrr
cus_ULWwH5W4Rtwl58,Active Starter #2,$21.00
cus_ULWwJAvHyIN6uT,Active Starter Heavy #3,$21.00
cus_ULWwNaOa2hh32z,Active Monthly Pro #4,$79.00
cus_ULWwO9uqpSPVUt,Upgraded Starter→Pro #9,$21.00
cus_ULWwQexwEAjgi3,Upgraded Starter→Pro #12,$21.00
cus_ULWwScgoS7pqiF,Churned Pro #11,$79.00
cus_ULWwTgCTM36v4e,Active Annual Enterprise #7,$207.50
cus_ULWwaIAUzeOUbH,Active Annual Pro #6,$65.83
cus_ULWwaKJiAP4F85,Active Monthly Pro #5,$79.00
cus_ULWwfk0VwelKjT,Downgraded Pro→Starter #10,$79.00


In [3]:
reports.churn.style_c_churned(detail)

customer,customer_name,starting_mrr,churned_mrr
cus_ULWwpTRPX25YZq,Churned Starter #8,$21.00,$21.00
cus_ULWx5rKuyz0Uin,Trial Jul →churn #18,$21.00,$21.00
cus_ULWxS396a8N2gD,Late Churned Starter #13,$21.00,$21.00
TOTAL,,$63.00,$63.00


## 2. Churn Rates

Logo and revenue churn from the Tidemill API.  These should match the
C_start / C_churned sets above:

- **Logo churn** = |C_churned| / |C_start|
- **Revenue churn** = sum(churned MRR) / sum(starting MRR)

In [4]:
reports.churn.style_snapshot(reports.churn.snapshot(tm, CHURN_START, END, detail=detail))

Metric,Numerator,Denominator,Rate
Logo churn,3 customers,17 customers,17.6%
Revenue churn,$142.00,$878.33,16.2%


### Revenue churn detail

Every customer from C_start with their starting MRR and churned MRR.
Customers who cancelled one subscription but kept another appear here
with churned MRR > 0 even though they are not in C_churned.

In [5]:
reports.churn.style_revenue_events(
    reports.churn.revenue_events(tm, CHURN_START, END, detail=detail)
)

customer,customer_name,starting_mrr,churned_mrr,fully_churned
cus_ULWwH5W4Rtwl58,Active Starter #2,$21.00,$0.00,False
cus_ULWwJAvHyIN6uT,Active Starter Heavy #3,$21.00,$0.00,False
cus_ULWwNaOa2hh32z,Active Monthly Pro #4,$79.00,$0.00,False
cus_ULWwO9uqpSPVUt,Upgraded Starter→Pro #9,$21.00,$0.00,False
cus_ULWwQexwEAjgi3,Upgraded Starter→Pro #12,$21.00,$0.00,False
cus_ULWwScgoS7pqiF,Churned Pro #11,$79.00,$79.00,False
cus_ULWwTgCTM36v4e,Active Annual Enterprise #7,$207.50,$0.00,False
cus_ULWwaIAUzeOUbH,Active Annual Pro #6,$65.83,$0.00,False
cus_ULWwaKJiAP4F85,Active Monthly Pro #5,$79.00,$0.00,False
cus_ULWwfk0VwelKjT,Downgraded Pro→Starter #10,$79.00,$0.00,False


## 3. Monthly Churn Rate Timeline

Compute logo and revenue churn rates per month to visualize churn trends over time.

In [6]:
churn_df = reports.churn.timeline(tm, CHURN_START, END)
reports.churn.style_timeline(churn_df)

,logo_churn,revenue_churn
month,,
2025-08,5.9%,2.4%
2025-09,15.0%,6.3%
2025-10,5.9%,2.4%
2025-11,5.0%,10.6%
2025-12,11.1%,5.0%
2026-01,8.3%,4.3%
2026-02,7.7%,4.2%


In [7]:
reports.churn.plot_timeline(churn_df)

## 4. Monthly Churned MRR

Lost MRR per month from the MRR waterfall — shows the dollar impact of churn over time.

In [8]:
reports.churn.plot_monthly_lost_mrr(reports.churn.monthly_lost_mrr(tm, START, END))